In [0]:
from pyspark.sql.functions import *
from pyspark.sql.window import Window

bronze_path = "abfss://bronze@travelappprojectstorage.dfs.core.windows.net/"
silver_path = "abfss://silver@travelappprojectstorage.dfs.core.windows.net/"
quarantine_path = "abfss://quarantine@travelappprojectstorage.dfs.core.windows.net/"

reading the bronze tables

In [0]:
df_alert = spark.read.parquet(bronze_path + "geofence_alert/")
df_place = spark.read.parquet(bronze_path + "danger_place/")
df_tourist = spark.read.parquet(bronze_path + "tourist/")
df_agency = spark.read.parquet(bronze_path + "agency/")
df_trip = spark.read.parquet(bronze_path + "trip/")

doing transformations


In [0]:
alert_cleaning = df_alert.dropDuplicates()
danger_place_cleaning = df_place.dropDuplicates()
tourist_cleaning = df_tourist.dropDuplicates()
agency_cleaning = df_agency.dropDuplicates()
trip_cleaning = df_trip.dropDuplicates()


In [0]:
invalid_alert_records = alert_cleaning.filter(col("AlertId").isNull())
valid_alert_records = alert_cleaning.filter(col("AlertId").isNotNull())

invalid_place_records = danger_place_cleaning.filter(col("DangerPlaceId").isNull())
valid_place_records = danger_place_cleaning.filter(col("DangerPlaceId").isNotNull())

invalid_tourist_records = tourist_cleaning.filter(col("TouristId").isNull())
valid_tourist_records = tourist_cleaning.filter(col("TouristId").isNotNull())

invalid_agency_records = agency_cleaning.filter(col("AgencyId").isNull())
valid_agency_records = agency_cleaning.filter(col("AgencyId").isNotNull())

invalid_trip_records = trip_cleaning.filter(col("TripId").isNull())
valid_trip_records = trip_cleaning.filter(col("TripId").isNotNull())

quarantine layer

In [0]:
invalid_alert_records.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .save(quarantine_path + "geofence_alert/")
invalid_place_records.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .save(quarantine_path + "danger_place/")
invalid_agency_records.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .save(quarantine_path + "agency/")
invalid_tourist_records.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .save(quarantine_path + "tourist/")
invalid_trip_records.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true")\
    .save(quarantine_path + "trip/")

In [0]:
other_transformations

In [0]:
null_value_alert_columns = ["TouristId", "PlaceId", "AlertId"]
df_alert_clean = alert_cleaning.dropna(subset=null_value_alert_columns)
df_alert_clean = df_alert_clean.fillna({
    "AlertType": "Unknown",
    "Severity": "Unknown",
    "Message": "Unknown",
    "IsResolved": False,
    "DistanceMeters": 0
})

null_value_place_columns = ["DangerPlaceId", "Name", "Latitude", "Longitude"]
df_place_clean = danger_place_cleaning.dropna(subset=null_value_place_columns)
df_place_clean = df_place_clean.fillna({
    "Description": "Unknown",
    "RadiusMeters": 0,
    "Severity": "Unknown",
    "AlertMessage": "Unknown"
})

null_value_tourist_columns = ["TouristId", "Name", "Contact"]
df_tourist_clean = tourist_cleaning.dropna(subset=null_value_tourist_columns)
df_tourist_clean = df_tourist_clean.fillna({
    "Gender": "Unknown",
    "Nationality": "Unknown",
    "Email": "Unknown",
    "KYCHash": "Unknown",
    "KycType": "Unknown",
    "EmergencyContact": "Unknown",
    "Address": "Unknown",
    "PasswordHash": "Unknown",
    "UserType": "Unknown"
})

null_value_agency_columns = ["AgencyId", "AgencyName", "OwnerName", "Contact"]
df_agency_clean = agency_cleaning.dropna(subset=null_value_agency_columns)
df_agency_clean = df_agency_clean.fillna({
    "EmailId": "Unknown",
    "LicenseNo": "Unknown",
    "LicenseURL": "Unknown",
    "AddressInfo": "Unknown",
    "PasswordHash": "Unknown"
})

null_value_trip_columns = ["TripId", "TouristId", "AgencyId"]
df_trip_clean = trip_cleaning.dropna(subset=null_value_trip_columns)
df_trip_clean = df_trip_clean.fillna({
    "AssignedEmployeeId": 0,
    "Status": "Unknown",
    "RequestId": 0
})

In [0]:
(
    df_alert_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path + "geofence_alert/")
)
(
    df_place_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path + "danger_place/")
)
(
    df_agency_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path + "agency/")
)
(
    df_tourist_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path + "tourist/")
)
(
    df_trip_clean.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(silver_path + "trip/")
)

In [0]:
displaying the overwritten tables

In [0]:
display(spark.read.format("delta").load(silver_path + "geofence_alert/"))